#  Grape Leaf Disease Classification using CNN and VGG16

This project aims to classify grape leaf diseases into 4 categories:
 'Black_rot', 'Esca_(Black_Measles)', 'Healthy', and 'Leaf_blight_(Isariopsis_Leaf_Spot)'.
 In this notebook, we train two models:
 1. A custom Convolutional Neural Network (CNN) built from scratch
 2. A Transfer Learning model using pre-trained VGG16
 The goal is to compare their performance on the same dataset.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
data_dir = "/kaggle/input/augmented-grape-disease-detection-dataset/Final Training Data"

In [ ]:
import os 

classes=os.listdir(data_dir)
print("Classes: ", classes)

class_counts= {}
for class_name in classes:
    class_path= os.path.join(data_dir, class_name)
    count= len(os.listdir(class_path))
    class_counts[class_name]= count

print("Number of images per class:")
for c, count in class_counts.items():
    print(f"{c:35}: {count}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

random_class = random.choice(classes)
sample_path= os.path.join(data_dir, random_class)
random_img= random.choice(os.listdir(sample_path))

img=cv2.imread(os.path.join(sample_path, random_img))
img_rgb= cv2.cvtColor(img, cv2.COLOR_BGR2RGB) #cv2 - bgr and matplotlib - rgb

plt.imshow(img_rgb)
plt.title(f"Sample Image - {random_class}")
plt.axis("off")
plt.show()


In [ ]:
#Image preprocessing and data loading
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_path = data_dir

img_size=224 #important size for VGG16
batch_size=32

datagen= ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator= datagen.flow_from_directory(
    base_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42
)

val_generator=datagen.flow_from_directory(
    base_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    shuffle=True,
    seed=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model= Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(img_size,img_size,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(256, (3,3), activation="relu"),
    Conv2D(512, (3,3), activation="relu"),
    Flatten(),
    Dropout(0.5),
    Dense(250, activation="relu"),
    Dropout(0.5),
    Dense(4, activation="softmax")
    ])

In [ ]:
model.compile(optimizer="adam",loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop= EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history=model.fit(train_generator,validation_data=val_generator,epochs=20,callbacks=[early_stop])

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.title("Accuracy Graph")
plt.show()

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Loss Graph")
plt.show()

In [ ]:
model.save("grape_classifier_model.h5")

In [ ]:
loss, acc=model.evaluate(val_generator)
print(f"Validation Accuracy: {acc:.2f}")

# Transfer Learning

In [ ]:
from tensorflow.keras.applications import VGG16

base_model=VGG16(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))

#freezing
for layer in base_model.layers:
    layer.trainable=False

In [ ]:
from tensorflow.keras.optimizers import Adam
model= Sequential([
    base_model,
    Flatten(),
    Dropout(0.5),
    Dense(250, activation="relu"),
    Dropout(0.5),
    Dense(4, activation="softmax")
])
model.compile(optimizer=Adam(learning_rate=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
#we add early stopping
history=model.fit(train_generator, validation_data=val_generator,epochs=20, callbacks=[early_stop])

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.title("Accuracy with VGG16")
plt.show()

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Loss with VGG16")
plt.show()

In [ ]:
model.save("grape_classifier_vgg16.h5")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [ ]:
#Prediction with shuffle=False

val_generator = datagen.flow_from_directory(
    base_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    shuffle=False  
)


In [ ]:
# Real classes
true_labels = val_generator.classes

# Predictions
pred_probs = model.predict(val_generator)
pred_labels = np.argmax(pred_probs, axis=1)  # one-hot - class


In [ ]:
cm = confusion_matrix(true_labels, pred_labels)
class_names = list(val_generator.class_indices.keys()) 

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
report = classification_report(true_labels, pred_labels, target_names=class_names)
print("Classification Report:")
print(report)



# Results Summary:

 The custom CNN model achieved a validation accuracy of approximately 97%. It successfully learned to distinguish between 4 classes of grape leaf images. In the second phase, we implemented a Transfer Learning approach using a pre-trained VGG16 model. This model slightly outperformed the custom CNN, reaching a validation accuracy of approximately 98%. The comparison indicates that transfer learning can provide a performance boost by leveraging learned features from large-scale datasets such as ImageNet.
However, it is worth noting that the custom CNN—despite learning from scratch—delivered a high and sufficient level of performance, and did so with faster training and lower computational cost compared to the VGG16-based model.

Resources

- ChatGPT – For explanations, debugging support, and model architecture suggestions.

- Course Materials & Old Projects – Lecture notes, previous image classification notebooks, and lab examples were used for guidance and inspiration.